# Memória dinâmica

*Rode uma célula com ▶ (ou Shift+Enter): o Sphinx compila e roda a célula no seu computador. Mude o código e rode de novo: este notebook é a sua cópia.*

**Depois desta unidade você consegue:**
- Alocar memória cujo tamanho só se conhece durante a execução, com malloc e calloc, conferir NULL e liberá-la.
- Aumentar um buffer com realloc, dobrando a capacidade, e explicar por que o ponteiro antigo não pode mais ser usado.
- Montar arrays de linhas com tamanhos diferentes, ou um bloco só indexado por linha e coluna.

Os **exercícios** deste notebook terminam com linhas `check(...)`: testes mínimos que imprimem ✓ quando o seu código está certo. Rode a próxima célula uma vez antes, para eles poderem usar o `check`.

In [ ]:
// shared: check_int e check_str imprimem ✓ ou ✗; os exercícios abaixo usam
void check_int(const char *what, long actual, long expected) {
    if (actual == expected) {
        printf("✓ %s\n", what);
    } else {
        printf("✗ %s: esperado %ld, veio %ld\n", what, expected, actual);
    }
}

void check_str(const char *what, const char *actual, const char *expected) {
    if (actual == NULL) {
        actual = "(NULL)";
    }
    if (strcmp(actual, expected) == 0) {
        printf("✓ %s\n", what);
    } else {
        printf("✗ %s: esperado \"%s\", veio \"%s\"\n", what, expected, actual);
    }
}

## Em resumo

Os arrays têm um tamanho fixado quando você escreve o código. Quando o tamanho vem dos dados, peça memória durante a execução: `malloc(bytes)` devolve um ponteiro para um bloco novo no **heap**, ou `NULL` quando não há. O bloco dura até você chamar `free`, mesmo depois que a função que o criou retorna.

#### 🎞️ malloc, realloc e free

![malloc reserva um bloco no heap, realloc o aumenta (talvez movendo-o) e free o devolve, enquanto o ponteiro guarda o endereço antigo até receber NULL](malloc-steps.pt-br.svg)

```c
#include <stdio.h>
#include <stdlib.h>

int main(void) {
    int capacity = 4, count = 0, value;
    int *values = malloc(capacity * sizeof *values);
    if (values == NULL) return 1;
    while (scanf("%d", &value) == 1) {
        if (count == capacity) {                 // cheio: dobra
            capacity *= 2;
            int *bigger = realloc(values, capacity * sizeof *values);
            if (bigger == NULL) { free(values); return 1; }
            values = bigger;
        }
        values[count++] = value;
    }
    printf("%d numbers\n", count);
    free(values);
    return 0;
}
```

O `realloc` pode mover o bloco, então continue usando o ponteiro que ele devolve.

**malloc, realloc e free:** o bloco cresce, mantém os valores e é liberado no fim.

In [ ]:
int n = 4;
int *squares = malloc(n * sizeof *squares);
if (squares == NULL) {
    return 1;
}
for (int i = 0; i < n; i++) {
    squares[i] = i * i;
}
int *bigger = realloc(squares, 2 * n * sizeof *squares);
if (bigger == NULL) {
    free(squares);
    return 1;
}
squares = bigger;              /* o bloco pode ter mudado de lugar */
for (int i = n; i < 2 * n; i++) {
    squares[i] = i * i;
}
for (int i = 0; i < 2 * n; i++) {
    printf("%d ", squares[i]);
}
printf("\n");
free(squares);

### ✏️ Exercício: Uma sequência no heap

Devolva um bloco novo de n ints com 0, 1, …, n − 1. Quem chama libera.

In [ ]:
int *range_array(int n) {
    // TODO: aloque n ints com malloc e preencha
    return calloc(n, sizeof(int));
}

int main(void) {
    int *r = range_array(4);
    check_int("r[0]", r[0], 0);
    check_int("r[3]", r[3], 3);
    free(r);
    return 0;
}

<details>
<summary>Ver uma resposta</summary>

```c
int *range_array(int n) {
    int *a = malloc(n * sizeof(int));
    if (a == NULL) {
        return NULL;
    }
    for (int i = 0; i < n; i++) {
        a[i] = i;
    }
    return a;
}

int main(void) {
    int *r = range_array(4);
    check_int("r[0]", r[0], 0);
    check_int("r[3]", r[3], 3);
    free(r);
    return 0;
}
```

</details>

**Cuidado:** depois de `free(p)`, `p` ainda guarda o endereço antigo. Usá-lo, ou liberá-lo duas vezes, é comportamento indefinido; as verificações de memória pegam isso.

### ✏️ Exercício: Duplique uma string

Devolva uma cópia de s num bloco novo do heap. Lembre do byte extra do `'\0'`.

In [ ]:
char *duplicate(const char *s) {
    // TODO: aloque strlen(s) + 1 bytes e use strcpy
    return calloc(1, 1);
}

int main(void) {
    char *d = duplicate("hello");
    check_str("a cópia", d, "hello");
    char *e = duplicate("");
    check_str("uma cópia vazia", e, "");
    free(d);
    free(e);
    return 0;
}

<details>
<summary>Ver uma resposta</summary>

```c
char *duplicate(const char *s) {
    char *copy = malloc(strlen(s) + 1);
    if (copy == NULL) {
        return NULL;
    }
    strcpy(copy, s);
    return copy;
}

int main(void) {
    char *d = duplicate("hello");
    check_str("a cópia", d, "hello");
    char *e = duplicate("");
    check_str("uma cópia vazia", e, "");
    free(d);
    free(e);
    return 0;
}
```

</details>

## Leituras

- [Manual Memory Allocation](https://beej.us/guide/bgc/html/split/manual-memory-allocation.html) (Beej's Guide to C Programming, chapter 12, 12 min): malloc e free, conferência de erros, calloc e realloc, inclusive "Reading in Lines of Arbitrary Length", que é o desafio Longest Line quase linha por linha.
- [Lecture 4: Copying and malloc](https://cs50.harvard.edu/x/2026/notes/4/#copying-and-malloc) (CS50x (Harvard), lecture notes, 10 min): Opcional: de "Copying and malloc" até "Garbage Values": por que copiar um ponteiro não copia a string, e a pilha e o heap numa figura.

## Pratique

O notebook cobriu o básico. Agora aplique nos desafios da unidade, na barra lateral do Sphinx, onde o Submit roda todos os testes, inclusive os ocultos:

- **Quantidade desconhecida** · Fácil
- **Linha mais longa** · Médio
- **Triângulo de Pascal** · Médio
- **Multiplicação de matrizes** · Difícil